In [24]:
import sys
print(sys.version)

3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [19]:
%pip install flask werkzeug

Note: you may need to restart the kernel to use updated packages.


In [26]:
import flask
import werkzeug

print("Flask version:", flask.__version__)
print("Flask installed successfully!")

Flask version: 3.1.3
Flask installed successfully!


C:\Users\haris\AppData\Local\Temp\ipykernel_3008\2435731803.py:4: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:", flask.__version__)


In [9]:
import os

folders = [
    "app",
    "app/templates",
    "app/uploads",
    "app/generated"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [21]:
requirements = """Flask
Werkzeug
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created!")

requirements.txt created!


In [5]:
html_code = """
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Compose Converter</title>

    <link
        href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css"
        rel="stylesheet"
    >
</head>

<body>

<div class="container mt-5">

    <div class="card shadow p-4">

        <h1 class="text-center mb-4">
            Docker Compose to Kubernetes Converter
        </h1>

        {% if error %}
        <div class="alert alert-danger">
            {{ error }}
        </div>
        {% endif %}

        <form method="POST" enctype="multipart/form-data">

            <div class="mb-3">

                <label class="form-label">
                    Select Docker Compose file
                </label>

                <input
                    type="file"
                    name="file"
                    class="form-control"
                    accept=".yml,.yaml"
                    required
                >

            </div>

            <div class="text-center">

                <button
                    type="submit"
                    class="btn btn-primary"
                >
                    Convert to Kubernetes
                </button>

            </div>

        </form>

    </div>

</div>

</body>
</html>
"""

with open("app/templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_code)

print("index.html created successfully!")

index.html created successfully!


In [22]:
main_code = r'''
import os
import subprocess

from flask import (
    Flask,
    request,
    render_template,
    send_file,
    redirect,
    url_for
)

from werkzeug.utils import secure_filename


app = Flask(__name__)


# -----------------------------------
# Project directories
# -----------------------------------

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

UPLOAD_FOLDER = os.path.join(BASE_DIR, "uploads")

OUTPUT_FOLDER = os.path.join(BASE_DIR, "generated")


# Create directories if they don't exist

os.makedirs(UPLOAD_FOLDER, exist_ok=True)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# -----------------------------------
# Home / Upload Route
# -----------------------------------

@app.route("/", methods=["GET", "POST"])
def upload():

    if request.method == "POST":

        # Check if file exists
        if "file" not in request.files:

            return render_template(
                "index.html",
                error="Please select a file."
            )


        file = request.files["file"]


        # Check filename
        if file.filename == "":

            return render_template(
                "index.html",
                error="Please select a file."
            )


        # Check extension
        if not file.filename.endswith((".yml", ".yaml")):

            return render_template(
                "index.html",
                error="Please upload a .yml or .yaml file."
            )


        # Secure filename
        filename = secure_filename(file.filename)


        # Save uploaded file
        filepath = os.path.join(
            UPLOAD_FOLDER,
            filename
        )

        file.save(filepath)


        # Output file
        output_file = os.path.join(
            OUTPUT_FOLDER,
            "k8s.yaml"
        )


        # -----------------------------------
        # Run Kompose
        # -----------------------------------

        try:

            result = subprocess.run(
                [
                    "kompose",
                    "convert",
                    "-f",
                    filepath,
                    "-o",
                    output_file
                ],
                capture_output=True,
                text=True
            )


            # Kompose error
            if result.returncode != 0:

                return render_template(
                    "index.html",
                    error=result.stderr
                )


        except FileNotFoundError:

            return render_template(
                "index.html",
                error="Kompose is not installed or not available in PATH."
            )


        # Redirect to download
        return redirect(
            url_for("download")
        )


    return render_template("index.html")


# -----------------------------------
# Download Route
# -----------------------------------

@app.route("/download")
def download():

    output_file = os.path.join(
        OUTPUT_FOLDER,
        "k8s.yaml"
    )


    if os.path.exists(output_file):

        return send_file(
            output_file,
            as_attachment=True,
            download_name="k8s.yaml"
        )


    return "Generated file not found.", 404


# -----------------------------------
# Run Flask
# -----------------------------------

if __name__ == "__main__":

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=True,
        use_reloader=False
    )
'''

with open("app/main.py", "w", encoding="utf-8") as f:
    f.write(main_code)

print("main.py created successfully!")

main.py created successfully!


In [23]:
import os

for root, dirs, files in os.walk("."):
    level = root.count(os.sep)
    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

./
    ComposeConverter.ipynb
    requirements.txt
    app/
        main.py
        generated/
            k8s.yaml
        templates/
            index.html
        uploads/
            test-compose.yml


In [11]:
import os

os.environ["PATH"] += os.pathsep + r"C:\Kompose"

print("Kompose path added successfully!")

Kompose path added successfully!


In [12]:
import subprocess

result = subprocess.run(
    ["kompose", "version"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

1.38.0 (a8f5d1cbd)




In [13]:
import subprocess

result = subprocess.run(
    ["kompose", "version"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

1.38.0 (a8f5d1cbd)




In [14]:
compose_file = """
services:

  web:
    image: nginx:latest
    ports:
      - "8080:80"
"""

with open(
    "app/uploads/test-compose.yml",
    "w"
) as f:
    f.write(compose_file)

print("Test Docker Compose file created!")

Test Docker Compose file created!


In [15]:
import subprocess

input_file = "app/uploads/test-compose.yml"

output_file = "app/generated/k8s.yaml"

result = subprocess.run(
    [
        "kompose",
        "convert",
        "-f",
        input_file,
        "-o",
        output_file
    ],
    capture_output=True,
    text=True
)

print("RETURN CODE:")
print(result.returncode)

print("\nOUTPUT:")
print(result.stdout)

print("\nERROR:")
print(result.stderr)

RETURN CODE:
0

OUTPUT:


ERROR:
INFO Kubernetes file "app/generated/k8s.yaml" created 



In [16]:
with open(
    "app/generated/k8s.yaml",
    "r"
) as f:

    kubernetes_yaml = f.read()

print(kubernetes_yaml)

---
apiVersion: v1
kind: Service
metadata:
  annotations:
    kompose.cmd: kompose convert -f app/uploads/test-compose.yml -o app/generated/k8s.yaml
    kompose.version: 1.38.0 (a8f5d1cbd)
  labels:
    io.kompose.service: web
  name: web
spec:
  ports:
    - name: "8080"
      port: 8080
      targetPort: 80
  selector:
    io.kompose.service: web

---
apiVersion: apps/v1
kind: Deployment
metadata:
  annotations:
    kompose.cmd: kompose convert -f app/uploads/test-compose.yml -o app/generated/k8s.yaml
    kompose.version: 1.38.0 (a8f5d1cbd)
  labels:
    io.kompose.service: web
  name: web
spec:
  replicas: 1
  selector:
    matchLabels:
      io.kompose.service: web
  template:
    metadata:
      annotations:
        kompose.cmd: kompose convert -f app/uploads/test-compose.yml -o app/generated/k8s.yaml
        kompose.version: 1.38.0 (a8f5d1cbd)
      labels:
        io.kompose.service: web
    spec:
      containers:
        - image: nginx:latest
          name: web
          port

In [27]:
import subprocess
import sys

flask_process = subprocess.Popen(
    [
        sys.executable,
        "app/main.py"
    ]
)

print("Flask server started!")
print("Open: http://127.0.0.1:5000")

Flask server started!
Open: http://127.0.0.1:5000
